# YOLOv4 on D-Fire: Darknet build and training

This notebook builds AlexeyAB Darknet on Google Colab and trains **YOLOv4** on the corrected D-Fire split
(classes: `smoke`, `fire`). Evaluation, plots and the speed benchmark are in `yolov4_test_evaluation.ipynb`.

| Setting | Value |
|---|---|
| Framework | AlexeyAB Darknet, built with GPU, cuDNN, half precision and OpenCV |
| Model | full YOLOv4 (`yolov4-fire-smoke.cfg`) |
| Input size | 416 x 416 |
| Batch size | 64 |
| Iterations | 18 298 (about 85 epochs over 13 777 training images) |
| Hardware | Google Colab, Tesla T4 |

Colab sessions are shorter than the full training time, so training ran over several sessions. Each new
session repeats sections 1 to 3 and then resumes from the last checkpoint in section 4.

**Files expected on Drive** under `Fire_smoke_detection/`: `configs/YOLOv4/obj.data`,
`configs/YOLOv4/yolov4-fire-smoke.cfg`, the image lists that `obj.data` points to, and
`experiments/YOLOv4/backup/` for checkpoints.

## 1. Check the GPU

In [ ]:
!nvidia-smi

Sun Jul 19 07:32:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [ ]:
# Mount Google Drive. The dataset, configs, weights and logs live there
# so that they survive Colab disconnects.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Build Darknet

### 3.1 Install the OpenCV development files

In [ ]:
# Darknet is compiled with OPENCV=1, which needs the OpenCV C++ headers.
# System packages are lost when the runtime restarts, so this runs every session.
!apt-get update -qq
!apt-get install -y libopencv-dev

# Confirm the installed OpenCV version
!pkg-config --modversion opencv4

### 3.2 Clone, configure and compile Darknet

In [ ]:
from pathlib import Path
import subprocess
import os
import shutil

# Darknet is cloned to /content, which is temporary. It is not kept on Drive
# because it can be re-cloned and rebuilt at the start of every session.
darknet_dir = Path("/content/darknet")

# Preserve an incomplete folder if the earlier clone failed
if darknet_dir.exists() and not (darknet_dir / ".git").exists():
    incomplete_dir = Path("/content/darknet_incomplete")

    if incomplete_dir.exists():
        shutil.rmtree(incomplete_dir)

    darknet_dir.rename(incomplete_dir)
    print("Preserved incomplete folder as:", incomplete_dir)

# Clone Darknet if necessary
if not (darknet_dir / ".git").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/AlexeyAB/darknet.git",
         str(darknet_dir)],
        check=True
    )
else:
    print("Darknet repository already exists.")

os.chdir(darknet_dir)

# Print the commit that was built, so the exact Darknet version is on record
print("\n===== DARKNET COMMIT =====")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

# Update Makefile settings
makefile = darknet_dir / "Makefile"
text = makefile.read_text()

# GPU=1         build with CUDA
# CUDNN=1       use cuDNN-accelerated layers
# CUDNN_HALF=1  mixed precision on the T4's Tensor Cores
# OPENCV=1      image loading through OpenCV
settings = {
    "GPU": "1",
    "CUDNN": "1",
    "CUDNN_HALF": "1",
    "OPENCV": "1",
}

# Replace each 'NAME=value' line in the Makefile
lines = text.splitlines()

for setting, value in settings.items():
    lines = [
        f"{setting}={value}" if line.startswith(f"{setting}=") else line
        for line in lines
    ]

makefile.write_text("\n".join(lines) + "\n")

print("\n===== BUILD SETTINGS =====")
for line in makefile.read_text().splitlines():
    if any(line.startswith(f"{name}=") for name in settings):
        print(line)

print("\n===== COMPILING DARKNET =====")
subprocess.run(["make", "-j2"], check=True)

executable = darknet_dir / "darknet"

print("\n===== COMPILE CHECK =====")
if executable.exists():
    print("SUCCESS: Darknet executable was created.")
    print("Location:", executable)
else:
    raise FileNotFoundError("Darknet executable was not created.")


===== DARKNET COMMIT =====

===== BUILD SETTINGS =====
GPU=1
CUDNN=1
CUDNN_HALF=1
OPENCV=1

===== COMPILING DARKNET =====

===== COMPILE CHECK =====
SUCCESS: Darknet executable was created.
Location: /content/darknet/darknet


## 4. Resume training from the last checkpoint

Continues from `yolov4-fire-smoke_last.weights`. Darknet saves this file as it trains, so a disconnected
session loses very little progress. Training is complete when the counter reaches 18298/18298.

In [ ]:
from pathlib import Path
import subprocess
from datetime import datetime


# --------------------------------------------------
# 1. Define the Darknet and project paths
# --------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Fire_smoke_detection")

DARKNET_DIR = Path("/content/darknet")
DARKNET = DARKNET_DIR / "darknet"

DATA_FILE = PROJECT / "configs/YOLOv4/obj.data"
CFG_FILE = PROJECT / "configs/YOLOv4/yolov4-fire-smoke.cfg"

RESUME_WEIGHTS = (
    PROJECT
    / "experiments/YOLOv4/backup"
    / "yolov4-fire-smoke_last.weights"
)

LOG_FILE = (
    PROJECT
    / "experiments/YOLOv4/logs"
    / "yolov4_training.log"
)


# --------------------------------------------------
# 2. Verify every input before starting
# --------------------------------------------------

print("===== RESUME PRECHECK =====")

for file in [
    DARKNET,
    DATA_FILE,
    CFG_FILE,
    RESUME_WEIGHTS,
]:
    print(file, "->", file.exists())

    if not file.exists():
        raise FileNotFoundError(file)

# A complete YOLOv4 checkpoint is about 244 MB. A much smaller file
# means the last save was interrupted and must not be resumed from.
checkpoint_size = (
    RESUME_WEIGHTS.stat().st_size / (1024 * 1024)
)

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MB",
)

if checkpoint_size < 200:
    raise RuntimeError(
        "The checkpoint appears incomplete."
    )


# --------------------------------------------------
# 3. Build the training command
# --------------------------------------------------
#
# stdbuf -oL -eL   line-buffer Darknet's output so progress appears live
# last.weights     Darknet continues from the iteration stored in the checkpoint
# -dont_show       no graphical window (Colab has no display)
# -map             calculate validation mAP during training and keep the
#                  best weights as yolov4-fire-smoke_best.weights
# --------------------------------------------------

print("\n===== RESUMING TRAINING =====")
print("Checkpoint:", RESUME_WEIGHTS)
print("Started:", datetime.now().isoformat())

command = [
    "stdbuf", "-oL", "-eL",
    str(DARKNET),
    "detector",
    "train",
    str(DATA_FILE),
    str(CFG_FILE),
    str(RESUME_WEIGHTS),
    "-dont_show",
    "-map",
]


# --------------------------------------------------
# 4. Run training and record every output line
# --------------------------------------------------
#
# The log is opened in append mode, so all sessions are kept in one file.
# The evaluation notebook later reads the validation history from it.
# --------------------------------------------------

with LOG_FILE.open("a") as log:
    log.write(
        "\n\n===== TRAINING RESUMED "
        f"{datetime.now().isoformat()} =====\n"
    )
    log.flush()

    process = subprocess.Popen(
        command,
        cwd=DARKNET_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\n===== PROCESS ENDED =====")
print("Darknet exit code:", exit_code)

[Output condensed for readability. The full console log is saved under experiments/YOLOv4/logs/]

===== RESUME PRECHECK =====
/content/darknet/darknet -> True
/content/drive/MyDrive/Fire_smoke_detection/configs/YOLOv4/obj.data -> True
/content/drive/MyDrive/Fire_smoke_detection/configs/YOLOv4/yolov4-fire-smoke.cfg -> True
/content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv4/backup/yolov4-fire-smoke_last.weights -> True
Checkpoint size: 244.18 MB

===== RESUMING TRAINING =====
Checkpoint: /content/drive/MyDrive/Fire_smoke_detection/experiments/YOLOv4/backup/yolov4-fire-smoke_last.weights
Started: 2026-07-19T07:35:34.433674
[network layer table and per-iteration loss lines omitted]

Last line before this session ended:
11106: 1.463840, 1.240584 avg loss, 0.001000 rate, 5.337236 seconds, 710784 images, 306.026215 hours left
